# Training ASL 24 Huruf: Hybrid SVM (1.867 Fitur)

Menggabungkan seluruh metode yang disyaratkan dalam satu pipeline:
1. **MediaPipe 21 Hand Landmarks** (deteksi ROI tangan + 52 fitur koordinat & jarak jari).
2. **Contour Detection** (`cv2.findContours` + *background suppression* di sekitar tangan).
3. **Ekstraksi Fitur Gabungan (1.867 dimensi)**:
   - **HOG**: 1.764 fitur (64×64, 9 arah, sel 8×8, blok 2×2, L2-Hys)
   - **Uniform LBP (2×2 grid)**: 40 fitur tekstur
   - **Hu Moments**: 7 fitur bentuk global
   - **Geometri Kontur**: 4 fitur (*aspect ratio, solidity, extent, circularity*)
   - **MediaPipe Landmarks**: 52 fitur (42 koordinat x,y ternormalisasi + 10 jarak ujung jari & telapak)
4. **RBF SVM + StandardScaler** lalu diekspor ke `svm_hog.bin.gz` (`SHG1`, `width=1867`).


In [ ]:
%pip -q install "numpy>=1.26,<3" "scikit-learn>=1.5,<2" "scikit-image>=0.24,<1" "opencv-python>=4.10,<6" "Pillow>=10,<14" "mediapipe>=0.10.14" "requests>=2.31,<3" "tqdm>=4.66,<5"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT_DRIVE = '/content/drive/MyDrive/LatihIsyarat_SVM_Hybrid1867'
# Opsional: folder foto kamera HP sendiri per huruf (folder/A/*.jpg, ...)
EXTERNAL_TRAIN_DIR = ''
MAX_SAMPLES_PER_CLASS = 250
INCLUDE_MIRROR_AUGMENTATION = True
SEED = 42


In [ ]:
import gzip, math, os, struct, zipfile
from pathlib import Path
import cv2
import mediapipe as mp
import numpy as np
import requests
from PIL import Image
from skimage.feature import hog
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm.auto import tqdm

LABEL_IDS = np.array([i for i in range(26) if i not in (9, 25)], dtype=np.int64)
LETTERS = [chr(65 + int(i)) for i in LABEL_IDS]

def _build_uniform_lbp_lut():
    lut = np.empty(256, dtype=np.int32)
    for code in range(256):
        transitions, ones = 0, 0
        for b in range(8):
            curr = (code >> b) & 1
            nxt = (code >> ((b + 1) & 7)) & 1
            if curr != nxt:
                transitions += 1
            ones += curr
        lut[code] = ones if transitions <= 2 else 9
    return lut

_LBP_LUT = _build_uniform_lbp_lut()

def lbp_grid_2x2(normalized):
    center = normalized[1:-1, 1:-1]
    offsets = [(-1, -1), (0, -1), (1, -1), (1, 0), (1, 1), (0, 1), (-1, 1), (-1, 0)]
    h, w = normalized.shape
    code = np.zeros((h - 2, w - 2), dtype=np.uint8)
    for b, (dx, dy) in enumerate(offsets):
        nbr = normalized[1 + dy:h - 1 + dy, 1 + dx:w - 1 + dx]
        code |= ((nbr >= center).astype(np.uint8) << b)
    bins = _LBP_LUT[code]
    out = np.zeros(40, dtype=np.float32)
    split = 31
    for q, patch in enumerate([bins[:split, :split], bins[:split, split:], bins[split:, :split], bins[split:, split:]]):
        hist = np.bincount(patch.ravel(), minlength=10).astype(np.float32)
        out[q * 10:(q + 1) * 10] = hist / float(patch.size or 1)
    return out

def _foreground_mask_64(normalized):
    border = np.concatenate((normalized[0, :], normalized[-1, :], normalized[:, 0], normalized[:, -1]))
    bg = int(np.median(border))
    delta = np.abs(normalized.astype(np.int32) - bg).clip(0, 255).astype(np.uint8)
    otsu_t, _ = cv2.threshold(delta, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return (delta >= max(int(otsu_t), 10)).astype(np.uint8)

def hu_moments_features(mask):
    ys, xs = np.nonzero(mask)
    m00 = float(len(xs))
    if m00 < 1.0:
        return np.zeros(7, dtype=np.float32)
    dx = xs.astype(np.float64) - float(xs.mean())
    dy = ys.astype(np.float64) - float(ys.mean())
    dx2, dy2 = dx * dx, dy * dy
    n2, n25 = m00 * m00, m00 ** 2.5
    nu20, nu02, nu11 = np.sum(dx2)/n2, np.sum(dy2)/n2, np.sum(dx*dy)/n2
    nu30, nu03, nu21, nu12 = np.sum(dx2*dx)/n25, np.sum(dy2*dy)/n25, np.sum(dx2*dy)/n25, np.sum(dx*dy2)/n25
    h1 = nu20 + nu02
    h2 = (nu20 - nu02)**2 + 4*nu11**2
    h3 = (nu30 - 3*nu12)**2 + (3*nu21 - nu03)**2
    h4 = (nu30 + nu12)**2 + (nu21 + nu03)**2
    h5 = (nu30 - 3*nu12)*(nu30 + nu12)*((nu30 + nu12)**2 - 3*(nu21 + nu03)**2) + (3*nu21 - nu03)*(nu21 + nu03)*(3*(nu30 + nu12)**2 - (nu21 + nu03)**2)
    h6 = (nu20 - nu02)*((nu30 + nu12)**2 - (nu21 + nu03)**2) + 4*nu11*(nu30 + nu12)*(nu21 + nu03)
    h7 = (3*nu21 - nu03)*(nu30 + nu12)*((nu30 + nu12)**2 - 3*(nu21 + nu03)**2) - (nu30 - 3*nu12)*(nu21 + nu03)*(3*(nu30 + nu12)**2 - (nu21 + nu03)**2)
    out = np.zeros(7, dtype=np.float32)
    for i, v in enumerate([h1, h2, h3, h4, h5, h6, h7]):
        if v != 0.0:
            out[i] = (-1.0 if v > 0 else 1.0) * math.log10(abs(v) + 1e-12)
    return out

def contour_geometry_features(mask):
    area = int(np.sum(mask > 0))
    if area == 0:
        return np.zeros(4, dtype=np.float32)
    ys, xs = np.nonzero(mask)
    bw = float(xs.max() - xs.min() + 1)
    bh = float(ys.max() - ys.min() + 1)
    padded = np.pad(mask, 1, mode='constant', constant_values=0)
    is_edge = (mask > 0) & ((padded[1:-1, :-2] == 0) | (padded[1:-1, 2:] == 0) | (padded[:-2, 1:-1] == 0) | (padded[2:, 1:-1] == 0))
    ey, ex = np.nonzero(is_edge)
    perimeter = len(ex)
    if perimeter < 3:
        return np.zeros(4, dtype=np.float32)
    pts = np.column_stack((ex, ey)).astype(np.int32)
    hull_area = float(cv2.contourArea(cv2.convexHull(pts)))
    return np.array([
        bw / bh,
        min(1.0, area / hull_area) if hull_area > 0 else 1.0,
        area / (bw * bh),
        (4.0 * math.pi * area) / (perimeter * perimeter),
    ], dtype=np.float32)

def extract_landmark_features(pts):
    wx, wy = pts[0]
    shifted = [(x - wx, y - wy) for x, y in pts]
    scale = max((math.hypot(dx, dy) for dx, dy in shifted[1:]), default=1.0)
    if scale < 1e-6:
        scale = 1.0
    norm = [(dx / scale, dy / scale) for dx, dy in shifted]
    feats = []
    for nx, ny in norm:
        feats.extend([nx, ny])
    for a, b in [(0, 4), (0, 8), (0, 12), (0, 16), (0, 20), (4, 8), (8, 12), (12, 16), (16, 20), (5, 17)]:
        feats.append(math.hypot(norm[a][0] - norm[b][0], norm[a][1] - norm[b][1]))
    return np.array(feats, dtype=np.float32)

def crop_roi_with_contour(gray, pts, margin_scale=1.10):
    h, w = gray.shape
    xs = [int(round(max(0.0, min(1.0, x)) * (w - 1))) for x, _ in pts]
    ys = [int(round(max(0.0, min(1.0, y)) * (h - 1))) for _, y in pts]
    x0, x1 = min(xs), max(xs)
    y0, y1 = min(ys), max(ys)
    side = max(x1 - x0 + 1, y1 - y0 + 1)
    size = max(2, math.ceil(side * margin_scale))
    left = (x0 + x1 + 1 - size) // 2
    top = (y0 + y1 + 1 - size) // 2
    fill = int(np.median(gray))
    canvas = Image.new('L', (size, size), fill)
    canvas.paste(Image.fromarray(gray), (-left, -top))
    small28 = canvas.resize((28, 28), Image.Resampling.BOX)
    img64 = small28.resize((64, 64), Image.Resampling.BILINEAR)
    return np.asarray(img64, dtype=np.uint8)

def extract_hybrid_1867(img64, pts):
    hog_vec = hog(img64.astype(np.float32) / 255.0, orientations=9, pixels_per_cell=(8, 8),
                  cells_per_block=(2, 2), block_norm='L2-Hys', transform_sqrt=True,
                  feature_vector=True, channel_axis=None).astype(np.float32)
    lbp_vec = lbp_grid_2x2(img64)
    mask = _foreground_mask_64(img64)
    hu_vec = hu_moments_features(mask)
    geo_vec = contour_geometry_features(mask)
    lm_vec = extract_landmark_features(pts)
    return np.concatenate((hog_vec, lbp_vec, hu_vec, geo_vec, lm_vec)).astype(np.float32)

def export_flutter_bin_gz(scaler, svm, output_path, landmark_weight=4.0):
    width, count, classes = int(scaler.mean_.shape[0]), int(svm.support_vectors_.shape[0]), int(svm.classes_.shape[0])
    gamma = float(svm._gamma)
    eff_scale = np.array(scaler.scale_, dtype=np.float32, copy=True)
    if width == 1867 and landmark_weight != 1.0:
        eff_scale[1815:] = eff_scale[1815:] / float(landmark_weight)
    payload = bytearray()
    payload += struct.pack('<4sIIId', b'SHG1', width, count, classes, gamma)
    payload += np.asarray(svm.classes_, dtype='<i4').tobytes()
    payload += np.asarray(svm.n_support_, dtype='<i4').tobytes()
    payload += np.asarray(scaler.mean_, dtype='<f4').tobytes()
    payload += np.asarray(eff_scale, dtype='<f4').tobytes()
    payload += np.asarray(svm.intercept_, dtype='<f4').tobytes()
    payload += np.asarray(svm.dual_coef_, dtype='<f4').tobytes()
    payload += np.asarray(svm.support_vectors_, dtype='<f4').tobytes()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(output_path, 'wb', compresslevel=6) as gz:
        gz.write(payload)
    print('Flutter asset exported:', output_path, f'({output_path.stat().st_size} bytes, width={width}, {count} SVs)')

root = Path(ROOT_DRIVE)
root.mkdir(parents=True, exist_ok=True)
dataset_zip = root / 'asl-alphabet.zip'
extract_dir = Path('/content/asl_alphabet_data')
if not extract_dir.exists():
    if not dataset_zip.exists():
        url = 'https://www.kaggle.com/api/v1/datasets/download/grassknoted/asl-alphabet'
        print('Mengunduh ASL Alphabet dataset ke Drive...')
        with requests.get(url, stream=True, timeout=(30, 300)) as r:
            r.raise_for_status()
            total = int(r.headers.get('Content-Length', 0)) or None
            with open(dataset_zip, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
    print('Mengekstrak dataset (hanya A..Y tanpa J/Z)...')
    with zipfile.ZipFile(dataset_zip) as zf:
        for info in tqdm(zf.infolist(), desc='Unzip'):
            parts = Path(info.filename).parts
            if any(p in LETTERS for p in parts) and not info.is_dir():
                zf.extract(info, extract_dir)
train_root = next(p.parent for p in extract_dir.rglob('A') if p.is_dir())
print('Dataset root:', train_root)


In [ ]:
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

task_path = root / 'hand_landmarker.task'
if not task_path.exists():
    task_url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
    print('Mengunduh model hand_landmarker.task...')
    with requests.get(task_url, timeout=60) as r:
        r.raise_for_status()
        task_path.write_bytes(r.content)

base_options = mp_python.BaseOptions(model_asset_path=str(task_path))
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.35,
)
detector = mp_vision.HandLandmarker.create_from_options(options)

rng = np.random.default_rng(SEED)
X_list, y_list = [], []

def process_folder(base_dir, max_per_class=None, label_tag='dataset'):
    for lid, letter in zip(LABEL_IDS, LETTERS):
        folder = Path(base_dir) / letter
        if not folder.is_dir():
            continue
        files = sorted([p for p in folder.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])
        if max_per_class and len(files) > max_per_class:
            idx = rng.choice(len(files), size=max_per_class, replace=False)
            files = [files[i] for i in sorted(idx)]
        ok = 0
        for p in files:
            bgr = cv2.imread(str(p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            res = detector.detect(mp_image)
            if not res.hand_landmarks:
                continue
            pts = [(lm.x, lm.y) for lm in res.hand_landmarks[0]]
            gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
            img64 = crop_roi_with_contour(gray, pts)
            X_list.append(extract_hybrid_1867(img64, pts))
            y_list.append(int(lid))
            ok += 1
            if INCLUDE_MIRROR_AUGMENTATION:
                pts_flip = [(1.0 - x, y) for x, y in pts]
                img64_flip = cv2.flip(img64, 1)
                X_list.append(extract_hybrid_1867(img64_flip, pts_flip))
                y_list.append(int(lid))
        print(f'[{label_tag}] {letter}: {ok}/{len(files)} gambar terdeteksi')

process_folder(train_root, max_per_class=MAX_SAMPLES_PER_CLASS, label_tag='ASL_Alphabet')
if EXTERNAL_TRAIN_DIR:
    process_folder(EXTERNAL_TRAIN_DIR, max_per_class=None, label_tag='Camera_HP')

detector.close()
X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
print('Total sampel Hybrid 1867:', X.shape)

LANDMARK_WEIGHT = 4.0  # Beri bobot 4x pada 52 fitur landmark agar tahan blur/noise video
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
tr_scaler = StandardScaler().fit(X_tr)
X_tr_s = tr_scaler.transform(X_tr)
X_val_s = tr_scaler.transform(X_val)
X_tr_s[:, 1815:] *= LANDMARK_WEIGHT
X_val_s[:, 1815:] *= LANDMARK_WEIGHT

best_f1, best_c = -1.0, 10.0
for c in [10.0, 50.0]:
    svm = SVC(C=c, gamma='scale', kernel='rbf')
    svm.fit(X_tr_s, y_tr)
    pred = svm.predict(X_val_s)
    acc, f1 = accuracy_score(y_val, pred), f1_score(y_val, pred, average='macro')
    print(f'C={c}: val_acc={acc*100:.2f}%, val_macro_f1={f1*100:.2f}%')
    if f1 > best_f1:
        best_f1, best_c = f1, c

final_scaler = StandardScaler().fit(X)
X_s = final_scaler.transform(X)
X_s[:, 1815:] *= LANDMARK_WEIGHT
final_svm = SVC(C=best_c, gamma='scale', kernel='rbf')
final_svm.fit(X_s, y)
out_bin = root / 'svm_hog.bin.gz'
export_flutter_bin_gz(final_scaler, final_svm, out_bin, landmark_weight=LANDMARK_WEIGHT)
from google.colab import files
files.download(str(out_bin))
